# Using RXIMO Explainer with River Pollution Problem

This notebook demonstrates how to use the RXIMO explainer to understand the relationship between reference points and objective values in the River Pollution problem. We'll use SHAP (SHapley Additive exPlanations) values to explain how changes in reference points affect the objective values.

## 1. Import Required Libraries

First, let's import the necessary libraries and modules.

In [3]:
import numpy as np
import polars as pl
from desdeo.explanations.explainer import ShapExplainer
from desdeo.explanations.utils import generate_biased_mean_data
import pandas as pd
import plotly.express as px

## 2. Load River Pollution Problem Data

We'll load the river pollution dataset and prepare it for the RXIMO explainer.

In [4]:
# Generate sample reference points
num_samples = 100
num_objectives = 4

# Generate random reference points between 0 and 1
np.random.seed(42)  # for reproducibility
reference_points = np.random.rand(num_samples, num_objectives)

# Define a simple achievement scalarizing function (ASF)
def achievement_scalarizing_function(f, ref_point, weights=None):
    if weights is None:
        weights = np.ones_like(ref_point)
    return np.max(weights * (f - ref_point))

In [5]:
# Define the River Pollution Problem functions
def river_pollution_objectives(x):
    """
    River Pollution Problem with 4 objectives
    x: array of shape (2,) containing [x1, x2]
    returns: array of shape (4,) containing [f1, f2, f3, f4]
    """
    x1, x2 = x[0], x[1]
    
    # Objective functions
    f1 = 4.07 + 2.27 * x1
    f2 = 2.60 + 0.03 * x1 + 0.02 * x2
    f3 = 8.21 - 0.71 * x1
    f4 = 0.96 - 0.96 * x2
    
    return np.array([f1, f2, f3, f4])

# Generate Pareto front approximation
def generate_pareto_solutions(num_points=100):
    solutions = []
    objectives = []
    
    # Sample decision variables
    x1_range = np.linspace(0, 1, int(np.sqrt(num_points)))
    x2_range = np.linspace(0, 1, int(np.sqrt(num_points)))
    
    for x1 in x1_range:
        for x2 in x2_range:
            x = np.array([x1, x2])
            f = river_pollution_objectives(x)
            solutions.append(x)
            objectives.append(f)
    
    return np.array(solutions), np.array(objectives)

# Generate solutions
solutions, objective_values = generate_pareto_solutions(num_points=100)

print("Generated solutions shape:", solutions.shape)
print("Generated objectives shape:", objective_values.shape)

Generated solutions shape: (100, 2)
Generated objectives shape: (100, 4)


In [6]:
# Solve for each reference point using ASF
def find_closest_solution(ref_point, objectives, weights=None):
    """Find the solution closest to the reference point using ASF"""
    asf_values = [achievement_scalarizing_function(obj, ref_point, weights) for obj in objectives]
    best_idx = np.argmin(asf_values)
    return objectives[best_idx]

# Generate dataset for RXIMO
dataset = []
for ref_point in reference_points:
    solution = find_closest_solution(ref_point, objective_values)
    dataset.append({
        'z_1': ref_point[0],
        'z_2': ref_point[1],
        'z_3': ref_point[2],
        'z_4': ref_point[3],
        'f_1': solution[0],
        'f_2': solution[1],
        'f_3': solution[2],
        'f_4': solution[3]
    })

# Create Polars DataFrame
pl_df = pl.DataFrame(dataset)

print("Generated dataset shape:", pl_df.shape)
print("\nFirst few rows:")
print(pl_df.head())

Generated dataset shape: (100, 8)

First few rows:
shape: (5, 8)
┌──────────┬──────────┬──────────┬──────────┬──────┬──────┬─────┬──────┐
│ z_1      ┆ z_2      ┆ z_3      ┆ z_4      ┆ f_1  ┆ f_2  ┆ f_3 ┆ f_4  │
│ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---  ┆ ---  ┆ --- ┆ ---  │
│ f64      ┆ f64      ┆ f64      ┆ f64      ┆ f64  ┆ f64  ┆ f64 ┆ f64  │
╞══════════╪══════════╪══════════╪══════════╪══════╪══════╪═════╪══════╡
│ 0.37454  ┆ 0.950714 ┆ 0.731994 ┆ 0.598658 ┆ 6.34 ┆ 2.63 ┆ 7.5 ┆ 0.96 │
│ 0.156019 ┆ 0.155995 ┆ 0.058084 ┆ 0.866176 ┆ 6.34 ┆ 2.63 ┆ 7.5 ┆ 0.96 │
│ 0.601115 ┆ 0.708073 ┆ 0.020584 ┆ 0.96991  ┆ 6.34 ┆ 2.63 ┆ 7.5 ┆ 0.96 │
│ 0.832443 ┆ 0.212339 ┆ 0.181825 ┆ 0.183405 ┆ 6.34 ┆ 2.63 ┆ 7.5 ┆ 0.96 │
│ 0.304242 ┆ 0.524756 ┆ 0.431945 ┆ 0.291229 ┆ 6.34 ┆ 2.63 ┆ 7.5 ┆ 0.96 │
└──────────┴──────────┴──────────┴──────────┴──────┴──────┴─────┴──────┘


## 3. Setup RXIMO Explainer

Now we'll set up the SHAP explainer with our data. For the river pollution problem, we have:
- Input symbols (z_1, z_2, z_3, z_4): Reference points for each objective
- Output symbols (f_1, f_2, f_3, f_4): Objective function values

In [7]:
# Define input and output symbols
input_symbols = [f'z_{i+1}' for i in range(4)]  # z_1 to z_4
output_symbols = [f'f_{i+1}' for i in range(4)]  # f_1 to f_4

# Create the SHAP explainer
explainer = ShapExplainer(
    problem_data=pl_df,
    input_symbols=input_symbols,
    output_symbols=output_symbols
)

print("Input symbols:", input_symbols)
print("Output symbols:", output_symbols)

Input symbols: ['z_1', 'z_2', 'z_3', 'z_4']
Output symbols: ['f_1', 'f_2', 'f_3', 'f_4']


## 4. Generate Background Data

We'll generate background data for the SHAP explainer. This data represents the baseline against which our explanations will be computed.

In [8]:
# Define a target reference point
target_ref_point = [0.5, 0.5, 0.5, 0.5]  # Example reference point

# Generate background data
try:
    background_indices = generate_biased_mean_data(
        pl_df[output_symbols].to_numpy(),
        target_ref_point,
        min_size=5,
        max_size=10
    )
    print("Successfully generated background data")
except Exception as e:
    print("Using fallback random sampling for background data")
    background_indices = np.random.choice(len(pl_df), size=10, replace=False)

# Create background dataset
background_data = pl_df.select(background_indices)
print("\nBackground data shape:", background_data.shape)

# Setup the explainer with background data
explainer.setup(background_data=background_data)

Successfully generated background data


DuplicateError: the name 'literal' is duplicate

It's possible that multiple expressions are returning the same default column name. If this is the case, try renaming the columns with `.alias("new_name")` to avoid duplicate column names.

## 5. Explain Reference Points

Now we can use the explainer to understand how different reference points influence the objective values.

In [ ]:
# Create a reference point to explain
ref_point_to_explain = {
    'z_1': 0.3,
    'z_2': 0.6,
    'z_3': 0.4,
    'z_4': 0.7
}

# Convert to Polars DataFrame
explain_df = pl.DataFrame([ref_point_to_explain])

# Get SHAP explanation
shap_values = explainer.explain_input(explain_df)

# Print SHAP values
print("SHAP Values shape:", shap_values.values.shape)
print("\nBase Values:", shap_values.base_values)
print("\nSHAP impact on objectives:")

# Calculate mean impact for each output
for i, output in enumerate(output_symbols):
    mean_impact = np.mean(shap_values.values[0, i, :])
    print(f"{output}: {mean_impact:.4f}")

## 6. Visualize SHAP Values

Let's create a visualization to better understand the relationships between reference points and objectives.

In [ ]:
# Prepare data for visualization
impacts = []
for i, output in enumerate(output_symbols):
    for j, input_sym in enumerate(input_symbols):
        impacts.append({
            'Output': output,
            'Input': input_sym,
            'Impact': shap_values.values[0, i, j]
        })

# Create DataFrame for plotting
impact_df = pd.DataFrame(impacts)

# Create heatmap
fig = px.imshow(
    shap_values.values[0],
    labels=dict(x="Input Reference Points", y="Output Objectives", color="SHAP Impact"),
    x=input_symbols,
    y=output_symbols,
    title="SHAP Values: Impact of Reference Points on Objectives",
    color_continuous_scale="RdBu"
)

fig.show()

## 7. Interpretation

The SHAP values and visualization show:
1. How each reference point (z_1 to z_4) influences each objective (f_1 to f_4)
2. The magnitude and direction of the influence (positive or negative)
3. The relative importance of different reference points for each objective

Red colors indicate positive impact (increasing the objective value), while blue colors indicate negative impact (decreasing the objective value).